In [34]:
import numpy as np
from numpy import random
from scipy.stats import norm
from scipy.stats import genextreme
import pandas as pd 
from numpy import linalg as la

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Data exploration

In [35]:
data = pd.read_csv('ppcs_cc.csv', skiprows=1, header=None)

columns = [
    'sblack', 'shisp', 'swhite', 'sother', 'smale', 'sage',
    'sempl', 'sincome', 'spop', 'daytime', 'inctype_lin',
    'omajblack', 'omajhisp', 'omajwhite', 'omajother',
    'osplit', 'sbehavior', 'year', 'anyuseofforce_coded'
]

data.columns = columns

In [36]:
print(data.head())

   sblack  shisp  swhite  sother  smale  sage  sempl  sincome  spop  daytime  \
0       1      0       0       0      1    18      0        1     1        1   
1       1      0       0       0      1    20      1        2     4        0   
2       1      0       0       0      1    22      1        2     3        1   
3       1      0       0       0      1    22      1        3     1        1   
4       1      0       0       0      1    22      1        1     1        1   

   inctype_lin  omajblack  omajhisp  omajwhite  omajother  osplit  sbehavior  \
0            2          0         0          1          0       0          0   
1            2          0         0          1          0       0          0   
2            2          0         0          1          0       0          0   
3            2          0         0          1          0       0          0   
4            2          0         0          1          0       0          0   

   year  anyuseofforce_coded  
0  2011

## Checking reference variabels

In [37]:
data[["sblack","shisp","swhite","sother"]].sum(axis=1).eq(1).all()

np.True_

In [38]:
data[[ 'omajblack', 'omajhisp', 'omajwhite', 'omajother','osplit']].sum(axis=1).eq(1).all()

np.True_

swhite and omajwhite are removed as reference categories

In [39]:
data2 = data.drop(columns=['swhite','omajwhite'])

Recoding inctype to dummy variabels

In [40]:
data2["Street_stop"] = (data2["inctype_lin"] == 1).astype(int)
data2["Traffic_stop"] = (data2["inctype_lin"] == 2).astype(int)
data2["Other_stop"] = (data2["inctype_lin"] == 3).astype(int)
data2=data2.drop(columns=['inctype_lin'])   

Checking reference categories

In [41]:
data2[["Street_stop", "Traffic_stop", "Other_stop"]].sum(axis=1).eq(1).all()

np.True_

Removing street_stop to be used as reference

In [42]:
data2=data2.drop(columns=['Street_stop'])

## Year variable 

From an initial look at the data it seems that all observations are from 2011 and therefore the year variable should not be included. This is first investegated

In [43]:
(data2["year"] == 2011).all()

np.True_

All observations are from 2011 so the year variable is deleted.

In [44]:
data2=data2.drop(columns=['year'])

## Cheking for other constant variabels

In [58]:
data2.nunique()

sblack                  2
shisp                   2
sother                  2
smale                   2
sage                   74
sempl                   2
sincome                 3
spop                    4
daytime                 2
omajblack               2
omajhisp                2
omajother               2
osplit                  1
sbehavior               2
anyuseofforce_coded     2
Traffic_stop            2
Other_stop              1
dtype: int64

In [59]:
print(data2["osplit"].unique())
print(data2["Other_stop"].unique())

[0]
[0]


Observe that both osplit and other_stop are constant (they never occur). We remove these from the dataset

In [60]:
data2=data2.drop(columns=['osplit','Other_stop'])

## Cheking missing values

In [61]:
data2.isna().sum().sum()

np.int64(0)

No missing values

## Defining X and Y and cheking rank of X

In [62]:
np.shape(data2)

(3799, 15)

In [63]:
X=data2.drop(columns=['anyuseofforce_coded'])
Y=data2['anyuseofforce_coded']
Xnames=X.columns
N=np.shape(X)[0]

In [64]:
np.shape(X)

(3799, 14)

In [65]:
print(la.matrix_rank(X))

14


X has full rank.